In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import SMOTE
import pickle, os

# Load the combined file saved by notebook 01
df = pd.read_csv("data/combined_raw.csv", low_memory=False)
label_col = [c for c in df.columns if 'label' in c.lower()][0]

print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Label column: '{label_col}'")

Loaded: 1,538,685 rows x 40 columns
Label column: 'label'


In [2]:
rows_before = len(df)
df = df.replace([np.inf, -np.inf], np.nan).dropna()
rows_after = len(df)

print(f"Before cleaning: {rows_before:,} rows")
print(f"After cleaning:  {rows_after:,} rows")
print(f"Removed:         {rows_before - rows_after:,} rows")

Before cleaning: 1,538,685 rows
After cleaning:  1,538,631 rows
Removed:         54 rows


In [3]:
label_map = {}
for lbl in df[label_col].unique():
    l = lbl.lower()
    if 'ddos' in l or ('dos' in l and 'recon' not in l):
        label_map[lbl] = 'DDoS'
    elif any(x in l for x in ['recon','scan','probe','sweep']):
        label_map[lbl] = 'Reconnaissance'
    elif any(x in l for x in ['benign','normal']):
        label_map[lbl] = 'Benign'

df['class'] = df[label_col].map(label_map)
df = df.dropna(subset=['class'])

print("Label mapping:")
for orig, mapped in sorted(label_map.items()):
    print(f"  {orig:<40} -> {mapped}")
print(f"Class distribution after mapping:")
print(df['class'].value_counts().to_string())

Label mapping:
  Benign                                   -> Benign
  DDoS                                     -> DDoS
  Reconnaissance                           -> Reconnaissance
Class distribution after mapping:
class
DDoS              798441
Benign            657907
Reconnaissance     82283


In [4]:
feature_cols = [c for c in df.columns if c not in [label_col, 'class']]
X = df[feature_cols]
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = MinMaxScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Features: {len(feature_cols)}")
print(f"Training: {len(X_train):,} samples")
print(f"Test:     {len(X_test):,} samples")

Features: 39
Training: 1,230,904 samples
Test:     307,727 samples


In [7]:
print("Before SMOTE (training set):")
print(y_train.value_counts().to_string())

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_sc, y_train)

print(f"After SMOTE (training set):")
print(pd.Series(y_train_bal).value_counts().to_string())
print(f"Total balanced training samples: {len(X_train_bal):,}")

Before SMOTE (training set):
class
DDoS              638753
Benign            526325
Reconnaissance     65826
After SMOTE (training set):
class
Benign            638753
DDoS              638753
Reconnaissance    638753
Total balanced training samples: 1,916,259


In [8]:
os.makedirs("data/processed", exist_ok=True)

pd.DataFrame(X_train_bal, columns=feature_cols).to_csv("data/processed/X_train.csv", index=False)
pd.DataFrame(X_test_sc,   columns=feature_cols).to_csv("data/processed/X_test.csv",  index=False)
pd.Series(y_train_bal, name='class').to_csv("data/processed/y_train.csv", index=False)
pd.Series(y_test,      name='class').to_csv("data/processed/y_test.csv",  index=False)

with open("data/processed/feature_cols.pkl", "wb") as f: pickle.dump(feature_cols, f)
with open("data/processed/scaler.pkl",       "wb") as f: pickle.dump(scaler, f)

print("Saved to data/processed/:")
print("  X_train.csv, X_test.csv, y_train.csv, y_test.csv")
print("  feature_cols.pkl, scaler.pkl")

Saved to data/processed/:
  X_train.csv, X_test.csv, y_train.csv, y_test.csv
  feature_cols.pkl, scaler.pkl
